# 离散数据的数值计算

学习目标：对采样数据计算差分、梯度、梯形积分与线性插值，正确处理间距、端点和范围；选学区分离散卷积与相关。

前置知识：数组、采样间隔、导数与定积分的基本含义。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

首个代码单元导入 NumPy，后续单元沿用 np。示例使用小型自制采样数据，坐标与数值一一对应。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 相邻变化量与变化率

每隔 2 秒记录一次位置，用 diff 求相邻位置的变化量，再除以相邻时间差，得到每个采样区间的平均变化率。

diff 只做相邻相减，不会自动除以采样间隔。对于长度为 N 的一维输入，一次差分得到 N − 1 个值，对应相邻点之间的区间，N 表示采样点个数。

In [1]:
import numpy as np

times = np.array([0.0, 2.0, 4.0, 6.0])
positions = np.array([0.0, 4.0, 12.0, 24.0])
changes = np.diff(positions)
rates = changes / np.diff(times)

print(changes)
print(rates, rates.shape)
# 变化量为 [4, 8, 12]；除以各段 2 秒，得到 [2, 4, 6]，形状为 (3,)。

[ 4.  8. 12.]
[2. 4. 6.] (3,)


diff 默认沿最后一轴计算，n 指连续做几次差分。二阶差分是对一阶差分再做差分，不会自动变成带物理单位的二阶导数。

下面两行分别表示两条采样序列，列表示 4 个采样点。axis=1 沿各行计算；改变 axis 会改变相减的方向。

In [2]:
samples = np.array([[0, 1, 4, 9], [10, 12, 16, 22]])
print(np.diff(samples, axis=1))  # 预期：两行分别为 [1 3 5]、[2 4 6]。
print(np.diff(samples, n=2, axis=1))
print(np.diff(samples, axis=0))
# 行内一阶差分为 [[1, 3, 5], [2, 4, 6]]，形状为 (2, 3)。
# 行内二阶差分为 [[2, 2], [2, 2]]；沿行轴相减得到 [[10, 11, 12, 13]]。
print(np.diff(np.array([5])))
# 只有一个采样点时，没有相邻区间，结果为空。

[[1 3 5]
 [2 4 6]]
[[2 2]
 [2 2]]
[[10 11 12 13]]
[]


无符号整数的差分仍是无符号整数。若下降量应为负数，需要在相减前转换到足够宽的有符号类型。不能在无符号差分已经发生后才转换。

In [3]:
readings = np.array([3, 1, 4], dtype=np.uint8)
unsigned_changes = np.diff(readings)
signed_changes = np.diff(readings.astype(np.int16))

print(unsigned_changes, unsigned_changes.dtype)  # 预期：[254 3] uint8，下降 2 在无符号类型中发生回绕。
print(signed_changes, signed_changes.dtype)
print(unsigned_changes.astype(np.int16))
# uint8 得到 [254, 3]；先转 int16 再相减才得到 [-2, 3]。
# 事后转换只把 [254, 3] 换成 int16，不能恢复原本的负变化量。

[254   3] uint8
[-2  3] int16
[254   3]


## 2 在采样点估计梯度

gradient 返回各采样点处的导数估计，输出与输入保持相同形状。它在内部点使用中心差分，在端点使用单边差分；默认采样间隔是 1，等距采样可直接传入实际间隔。

用 y = x² 的样本核对，已知导数为 2x。下面的间隔为 0.5；省略间隔会把数据当作间隔为 1 的样本。

In [4]:
x = np.array([0.0, 0.5, 1.0, 1.5])
y = x ** 2

print(np.gradient(y, 0.5))  # 预期：[0.5 1.0 2.0 2.5]。
print(np.gradient(y))
print(2 * x)
# 正确传入间距时为 [0.5, 1, 2, 2.5]；省略后为 [0.25, 0.5, 1, 1.25]。
# 导数基准为 [0, 1, 2, 3]；默认端点估计与基准不同。
print(np.gradient(y, 0.5).shape)
# 形状仍为 (4,)，与 diff 得到的区间数量不同。

[0.5 1.  2.  2.5]
[0.25 0.5  1.   1.25]
[0. 1. 2. 3.]
(4,)


### 2.1 端点公式与采样数量

edge_order 选择端点差分公式，支持 1 和 2，默认是 1；它不改变内部点的处理方式。计算轴至少需要 edge_order + 1 个点，因此 edge_order=2 至少需要 3 个采样点。

对这个二次函数，使用 edge_order=2 可与导数基准核对；这不意味着对任意数据都能得到精确导数。

In [5]:
x = np.array([0.0, 0.5, 1.0, 1.5])
y = x ** 2
first_edges = np.gradient(y, x, edge_order=1)
second_edges = np.gradient(y, x, edge_order=2)

print(first_edges)  # 预期：[0.5 1.0 2.0 2.5]，两端使用一阶边界公式。
print(second_edges)
print(second_edges - 2 * x)
# 二阶端点公式得到 [0, 1, 2, 3]，本例与基准的差为全零。

# 预期 ValueError：edge_order=2 至少需要三个点，此处只有两个。
np.gradient(np.array([0.0, 1.0]), edge_order=2)

[0.5 1.  2.  2.5]
[0. 1. 2. 3.]
[0. 0. 0. 0.]


ValueError: Shape of array too small to calculate a numerical gradient, at least (edge_order + 1) elements are required.

### 2.2 不等距坐标与轴

不等距采样时，向 gradient 传入完整坐标数组，长度必须与对应计算轴一致。传入的是坐标，不是 np.diff 得到的间隔数组。本章采用严格递增且有限的坐标，避免相邻点重合。

多维数据用 axis 指定计算方向。下面每行是一条曲线，每列对应一个 x 坐标，只沿列方向计算。

In [6]:
x = np.array([0.0, 0.5, 1.5, 3.0])
samples = np.stack([x ** 2, 2 * x ** 2])
gradients = np.gradient(samples, x, axis=1, edge_order=2)
expected = np.stack([2 * x, 4 * x])

print(gradients, gradients.shape)
print(np.max(np.abs(gradients - expected)))
# 两行分别接近 [0, 1, 3, 6]、[0, 2, 6, 12]，形状为 (2, 4)。
print(np.allclose(gradients, expected, rtol=0, atol=1e-12))
# 基准最大为 12；这里用 1e-12 绝对容差容纳 float64 运算舍入，不表示算法误差上界。

# 预期 ValueError：坐标参数需与四个采样点等长，np.diff(x) 只有三个间隔。
np.gradient(samples, np.diff(x), axis=1)

[[ 0.  1.  3.  6.]
 [ 0.  2.  6. 12.]] (2, 4)
0.0
True


ValueError: when 1d, distances must match the length of the corresponding dimension

## 3 用梯形面积近似积分

trapezoid 将相邻采样点连成直线，对每段梯形面积求和。一段的贡献是“两个端点值的平均数 × 坐标差”。等距采样可传 dx；未给 x 或 dx 时默认间隔为 1，不等距采样应给 x。

对 y = x² 在 x=0、1、3 的采样，第二段宽度是第一段的两倍。先看左图：每段面积依赖高度，也依赖横坐标差。

![左图对不等距样本的分段直线下方求面积，右图在同一折线上读插值高度；插值不等于原函数。](image/illustration/23-01-samples-area-interpolation.svg)

蓝线连接已知样本；左图是本节的面积计算，右图预示第 5 节从同一条折线读取新位置高度。右图紫色虚线是本例已知的 y=x²，用于比较，不是 API 根据样本恢复出的曲线。

下面只计算左图的两段贡献，先手算面积，再与 API 对照。省略 x 相当于改变采样间距，不能当作同一次积分。

In [7]:
x = np.array([0.0, 1.0, 3.0])
y = x ** 2
segments = (y[:-1] + y[1:]) / 2 * np.diff(x)

print(segments, segments.sum())  # 预期：分段面积为 [0.5 10.0]，总和为 10.5。
print(np.trapezoid(y, x=x))
print(np.trapezoid(y))
# 两段贡献为 [0.5, 10]，总和为 10.5。
# 省略 x 会按单位间隔计算，得到 5.5，描述的是另一组采样条件。

[ 0.5 10. ] 10.5
10.5
5.5


对于等距采样，给定 x 和给定 dx 可以表达相同的间隔。多条曲线可用 axis 指定积分轴，结果去掉这个轴。

In [8]:
x = np.array([0.0, 0.5, 1.0])
samples = np.stack([x ** 2, 2 * x ** 2])

print(np.trapezoid(samples, x=x, axis=1))
print(np.trapezoid(samples, dx=0.5, axis=1))
# 两种写法都得到 [0.375, 0.75]，输入 (2, 3) 沿轴 1 积分后为 (2,)。

[0.375 0.75 ]
[0.375 0.75 ]


trapezoid 按 x 的给定顺序计算，不会排序。反向经过相同采样点会改变积分方向；来回移动的坐标则按每段坐标差累加。是否需要排序由数据含义决定，不能只排序坐标而不处理对应的 y。

In [9]:
x = np.array([0.0, 1.0, 2.0])
y = x ** 2
print(np.trapezoid(y, x=x))
print(np.trapezoid(y[::-1], x=x[::-1]))
# 正向为 3，反向为 -3。

path_x = np.array([0.0, 2.0, 1.0])
path_y = path_x ** 2
print(np.trapezoid(path_y, x=path_x))
print((0 + 4) / 2 * 2 + (4 + 1) / 2 * (-1))
# 按 0 → 2 → 1 的顺序得到 1.5；函数没有把点自动重排为 0、1、2。

3.0
-3.0
1.5
1.5


## 4 改变间距观察误差

已知 x² 在 [0, 1] 上的定积分为 1/3，导数为 2x。用相同函数、相同区间比较 3 点与 5 点采样，分别观察梯形积分误差和默认端点梯度误差。

这里打印相对于已知答案的误差；它包含当前近似方法与浮点运算的影响。这个小例子的变化趋势不能作为任意数据的误差保证。

In [10]:
for count in (3, 5):
    x = np.linspace(0.0, 1.0, count)
    y = x ** 2
    area = np.trapezoid(y, x=x)
    gradient = np.gradient(y, x, edge_order=1)
    print("间距：", x[1] - x[0])  # 预期：两轮依次为 0.5、0.25。
    print("积分与绝对误差：", area, abs(area - 1 / 3))
    print("最大梯度误差：", np.max(np.abs(gradient - 2 * x)))
# 间距 0.5、0.25 时，积分分别为 0.375、0.34375，误差约 0.04167、0.01042。
# 最大梯度误差分别为 0.5、0.25；本例减小间距后两项误差都减小。

间距： 0.5
积分与绝对误差： 0.375 0.041666666666666685
最大梯度误差： 0.5
间距： 0.25
积分与绝对误差： 0.34375 0.010416666666666685
最大梯度误差： 0.25


## 5 在新坐标处线性插值

interp 根据已知坐标 xp 和已知值 fp，用相邻点之间的直线估计查询坐标 x 处的值。xp 和 fp 必须一维且等长；本章不设置 period，因此 xp 必须严格递增，不能包含 NaN。

回看第 3 节图的右半部分：查询位置落在两个样本之间，就读取它们连线上的高度。x=2 时折线上为 5，而已知原函数 x² 为 4；插值连接样本，不保证恢复原函数。

下面复用图中的不等距坐标，分别查询 0.5 和 2。先判断各自落在哪一段，再比较 interpolated 与 query**2，并核对输出形状与 query 相同。

In [11]:
xp = np.array([0.0, 1.0, 3.0])
fp = xp ** 2
query = np.array([0.5, 2.0])
interpolated = np.interp(query, xp, fp)

print(interpolated, interpolated.shape)
print(query ** 2)
# 两个查询点分别处在相邻样本中间，线性插值得到 [0.5, 5]，形状为 (2,)。
# 原函数 x² 在查询点的值为 [0.25, 4]，并不等于线性插值结果。

[0.5 5. ] (2,)
[0.25 4.  ]


### 5.1 插值范围之外

默认情况下，小于 xp[0] 的查询返回 fp[0]，大于 xp[-1] 的查询返回 fp[-1]。这不是按末段直线继续外推。用 left 和 right 可以指定范围外的返回值，例如用 NaN 标记超出范围。

In [12]:
xp = np.array([0.0, 1.0, 3.0])
fp = np.array([0.0, 1.0, 9.0])
query = np.array([-1.0, 0.0, 2.0, 3.0, 4.0])

print(np.interp(query, xp, fp))
print(np.interp(query, xp, fp, left=np.nan, right=np.nan))
# 默认得到 [0, 0, 5, 9, 9]；自定义后只有两个范围外位置为 NaN。
# 等于左右端点的查询仍返回端点样本值。

[0. 0. 5. 9. 9.]
[nan  0.  5.  9. nan]


### 5.2 输入条件要先检查

interp 不会严格执行 xp 递增检查，不能把“没有异常”当成输入合法。下面检查严格递增条件；重复坐标也不满足条件。长度不一致则是 API 会拒绝的输入。

In [13]:
xp = np.array([0.0, 2.0, 1.0])
duplicate_xp = np.array([0.0, 1.0, 1.0])
print(np.all(np.diff(xp) > 0))
print(np.all(np.diff(duplicate_xp) > 0))
# 都为 False；应先确定数据顺序或重复点处理规则，不把它们直接交给 interp。

# 预期 ValueError：xp 有两个坐标，fp 却只有一个对应值，两者长度不一致。
np.interp([0.5], [0.0, 1.0], [2.0])

False
False


ValueError: fp and xp are not of the same length.

## 6 选学：离散卷积
convolve 计算两个一维序列的离散线性卷积。对序列 a、v，输出 c 在位置 n 的定义是：

$$c_n = \sum_m a_m v_{n-m}$$

m 是求和索引，n 是输出索引；有限序列之外按零处理。可理解为将第二个序列反转，再滑动对齐并逐项相乘求和；这里的卷积符号不表示 ndarray 的逐元素乘法。

先对长度为 3 与 2 的短序列手算完整结果。

In [14]:
a = np.array([1, 2, 3])
v = np.array([1, 2])
manual = np.array([1 * 1, 1 * 2 + 2 * 1, 2 * 2 + 3 * 1, 3 * 2])

print(manual)
print(np.convolve(a, v, mode="full"))
# 完整卷积为 [1, 4, 7, 6]；两端只有一对非零位置重叠。

[1 4 7 6]
[1 4 7 6]


mode 决定保留哪些重叠位置。设输入长度为 N、M，两者均为正整数。

| mode | 中文含义 | 输出长度与端点 |
| --- | --- | --- |
| full | 完整结果 | N + M − 1；包含部分重叠位置，是 convolve 的默认值 |
| same | 保留中部结果 | max(N, M)；仍包含边界效应，不保证每处都有完整窗口 |
| valid | 完全重叠结果 | max(N, M) − min(N, M) + 1；只保留短序列完全重叠的位置 |

下面对常数序列做宽度为 3 的平均。same 的端点会把边界外的零计入，不能把输出长度相同理解为边界无需处理。

In [15]:
signal = np.ones(5)
kernel = np.ones(3) / 3
for mode in ("full", "same", "valid"):
    result = np.convolve(signal, kernel, mode=mode)
    print(mode, np.round(result, 6), result.shape)
# full 长度为 7；same 长度为 5，两端约为 0.666667，中间为 1。
# valid 长度为 3，全部为 1；这里只观察完整窗口，没有补出两端的估计。

full

 [0.333333 0.666667 1.       1.       1.       0.666667 0.333333] (7,)
same [0.666667 1.       1.       1.       0.666667] (5,)
valid [1. 1. 1.] (3,)


## 7 选学：离散相关
correlate 采用下面的互相关定义：

$$c_k = \sum_n a_{n+k}\,\overline{v_n}$$

k 表示相对位移，n 是求和索引，上横线表示复共轭；序列范围之外按零处理。相关的位移约定并不唯一，这里使用 NumPy 的定义。它与卷积的索引方向不同，不能视为同一种计算，也不是自动归一化的相关系数。

correlate 的 mode 含义与 convolve 相同，但默认值是 valid。下面完整结果的位移依次为 −1、0、1、2。

In [16]:
a = np.array([1, 2, 3])
v = np.array([1, 2])
manual = np.array([1 * 2, 1 * 1 + 2 * 2, 2 * 1 + 3 * 2, 3 * 1])

print(manual)  # 预期：[2 5 8 3]。
print(np.correlate(a, v, mode="full"))  # 预期：[2 5 8 3]，与手算一致。
print(np.correlate(a, v))
print(np.convolve(a, v, mode="full"))
# 相关完整结果为 [2, 5, 8, 3]；默认 valid 只保留 [5, 8]。
# 相同输入的卷积为 [1, 4, 7, 6]，二者不同。

[2 5 8 3]


[2 5 8 3]
[5 8]
[1 4 7 6]


复数相关会对第二个输入取共轭，卷积不会。按上面的定义，相关可与“第二序列先反转并取共轭，再卷积”的结果核对。

In [17]:
a = np.array([1 + 1j, 2])
v = np.array([1j, 1])

print(np.correlate(a, v, mode="valid"))
print((1 + 1j) * (-1j) + 2 * 1)
# 完全重叠时为 3 - 1j：第二序列中的 1j 先变成 -1j。
print(np.correlate(a, v, mode="full"))
print(np.convolve(a, np.conjugate(v[::-1]), mode="full"))
# 两种写法得到 [1 + 1j, 3 - 1j, -2j]。

[3.-1.j]
(3-1j)
[1.+1.j 3.-1.j 0.-2.j]
[1.+1.j 3.-1.j 0.-2.j]


## 本章小结

（1）diff 返回相邻区间的变化量；需要变化率时还要除以间隔，并先确认 dtype 能否表示负值。

（2）gradient 在采样点估计导数，需提供实际间距或坐标；端点公式与点数约束要一起考虑。

（3）trapezoid 按给定 x 的顺序计算；interp 需要递增坐标，且范围外默认返回端点值。

（4）卷积与相关的索引方向、复共轭和默认 mode 不同；输出长度相同不代表端点都有完整数据支持。

## 练习

（1）下降量与采样间距。记录使用 uint8 存储，采样时间间隔并不相同。先选择能表示负差值的类型，再计算相邻变化量与平均变化率；说明为什么不能先做 uint8 差分再转换类型。

In [18]:
times = np.array([0.0, 1.0, 3.0, 6.0])
readings = np.array([8, 5, 9, 3], dtype=np.uint8)
# 在此转换类型，计算并打印变化量、变化率和形状。
# 检查：结果对应 3 个区间；除数来自各区间实际时间差。

（2）采样条件改变。y = x² 原来等距采样，现在改用下面的不等距坐标。选择 gradient 的间距参数与 edge_order，并说明理由；与已知导数 2x 核对。若只剩两个点，参数或数据需要怎样调整？

In [19]:
x = np.array([0.0, 0.25, 1.0, 2.0])
y = x ** 2
expected = 2 * x
# 在此计算梯度，打印结果、与 expected 的差和所选参数。
# 检查：使用完整坐标；说明端点处理和最少点数，不只报告数值接近。

（3）积分方向与插值范围。对下面采样点计算梯形积分，再把 x、y 同时反向后重算。对 query 做线性插值，要求范围外返回 NaN；解释为什么反向输入可以直接用于此处的积分，却不能直接作为本章 interp 的 xp。

In [20]:
x = np.array([0.0, 1.0, 2.5])
y = np.array([1.0, 3.0, 4.0])
query = np.array([-0.5, 0.5, 2.5, 3.0])
# 在此积分并插值。
# 检查：手算每段梯形贡献；区分端点位置与范围外位置。

（4）卷积与相关（选学）。对给定短序列分别手算 full 模式的卷积和相关，再用 NumPy 核对。指出 valid 保留哪些完整重叠位置，并说明两个 API 的默认模式是否相同。

In [21]:
a = np.array([2, 1, 3])
v = np.array([1, -1])
# 在此写出手算过程，再调用 np.convolve 和 np.correlate。
# 检查：明确指定 mode；不要因为输入是实数就假定卷积与相关相同。

### 重点练习提示

对应第（2）题。先独立完成，再按需要查看提示。

（1）坐标间距不等时，一个固定标量步长不足以描述输入。

（2）向 gradient 传入完整 x；若希望二次函数的端点也按二阶差分核对，检查 edge_order 与点数。

### 重点练习参考解析

对应第（2）题。

选择 gradient(y, x, edge_order=2)，传入各点实际坐标。对本题二次函数，结果在浮点容差内为 [0, 0.5, 2, 4]，即 2x，形状仍为 (4,)。不能把默认步长 1 当作这些不等距坐标的间隔。

二阶端点需要至少 3 个点；只剩 2 个点时，可改用 edge_order=1 得到这两点之间的割线斜率，或补采样点后保留二阶端点。前一种选择能运行，但不再保证二次函数两端的导数均精确，不能把降低要求写成完全等价替代。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [diff](https://numpy.org/doc/2.5/reference/generated/numpy.diff.html) 的定义、Parameters、Notes：差分、axis、n 与无符号类型；[gradient](https://numpy.org/doc/2.5/reference/generated/numpy.gradient.html) 的 Parameters、Returns、Notes 与 Examples：实际间距、不等距坐标、端点和轴；[trapezoid](https://numpy.org/doc/2.5/reference/generated/numpy.trapezoid.html) 的定义、Parameters、Examples：给定坐标顺序、dx、axis、反向积分及 x² 在 [0, 1] 上积分为 1/3 的基准；[interp](https://numpy.org/doc/2.5/reference/generated/numpy.interp.html) 的 Parameters、Raises、Warning：递增输入、长度、NaN 与左右范围；[convolve](https://numpy.org/doc/2.5/reference/generated/numpy.convolve.html) 的 Parameters、Notes、Examples：离散定义、mode 与边界；[correlate](https://numpy.org/doc/2.5/reference/generated/numpy.correlate.html) 的定义、Parameters、Notes、Examples：位移约定、复共轭及默认模式；[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的 Parameters、Notes：绝对与相对容差的比较规则。 |
| GitHub（NumPy 官方源码，v2.5.0） | [numpy/lib/_function_base_impl.py](https://github.com/numpy/numpy/blob/v2.5.0/numpy/lib/_function_base_impl.py)：gradient 中计算轴长度与 edge_order + 1 的检查、坐标长度检查；trapezoid 中 diff(x) 与相邻 y 平均值乘坐标差的实现。 |
| MIT OpenCourseWare | [8.01SC，1.6 Derivatives](https://ocw.mit.edu/courses/8-01sc-classical-mechanics-fall-2016/pages/week-1-kinematics/1-6-derivatives/) 的 List of useful derivatives / Derivative of a polynomial function：用于核对二次函数梯度的幂函数求导规则。 |